In [1]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

builder = (
    SparkSession.builder
    .appName("Olist")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [2]:
spark.sql("CREATE DATABASE IF NOT EXISTS olist_delta")
spark.sql("USE olist_delta")

print("Base de dados olist_delta criada!")

Base de dados olist_delta criada!


In [3]:
# 1. SILVER_ORDERS
spark.sql("""
CREATE TABLE IF NOT EXISTS olist_delta.silver_orders (
    order_id STRING,
    customer_id STRING,
    order_status STRING,
    order_purchase_timestamp TIMESTAMP,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP
)
USING DELTA
""")

# 2. SILVER_ORDER_ITEMS
spark.sql("""
CREATE TABLE IF NOT EXISTS olist_delta.silver_order_items (
    order_id STRING,
    order_item_id INT,
    product_id STRING,
    seller_id STRING,
    shipping_limit_date TIMESTAMP,
    price DOUBLE,
    freight_value DOUBLE
)
USING DELTA
""")

# 3. SILVER_ORDER_REVIEWS
spark.sql("""
CREATE TABLE IF NOT EXISTS olist_delta.silver_order_reviews (
    review_id STRING,
    order_id STRING,
    review_score INT,
    review_comment_title STRING,
    review_comment_message STRING,
    review_creation_date TIMESTAMP,
    review_answer_timestamp TIMESTAMP
)
USING DELTA
""")

# 4. SILVER_SELLERS
spark.sql("""
CREATE TABLE IF NOT EXISTS olist_delta.silver_sellers (
    seller_id STRING,
    seller_zip_code_prefix STRING,
    seller_city STRING,
    seller_state STRING
)
USING DELTA
""")

# 5. SILVER_CUSTOMERS
spark.sql("""
CREATE TABLE IF NOT EXISTS olist_delta.silver_customers (
    customer_id STRING,
    customer_unique_id STRING,
    customer_zip_code_prefix STRING,
    customer_city STRING,
    customer_state STRING
)
USING DELTA
""")

# 6. SILVER_WEATHER_OBSERVATIONS
spark.sql("""
CREATE TABLE IF NOT EXISTS olist_delta.silver_weather_observations (
    station_id STRING,
    state_code STRING,
    observation_date DATE,
    observation_time STRING,
    precipitation DOUBLE,
    temperature DOUBLE,
    relative_humidity DOUBLE,
    latitude DOUBLE,
    longitude DOUBLE
)
USING DELTA
""")

# 7. SILVER_WEATHER_DAILY_STATE
spark.sql("""
CREATE TABLE IF NOT EXISTS olist_delta.silver_weather_daily_state (
    state_code STRING,
    observation_date DATE,
    avg_precipitation DOUBLE,
    avg_temperature DOUBLE,
    avg_relative_humidity DOUBLE,
    station_count INT,
    precipitation_coverage DOUBLE
)
USING DELTA
""")

# 8. GOLD_SELLER_PERFORMANCE
spark.sql("""
CREATE TABLE IF NOT EXISTS olist_delta.gold_seller_performance (
    seller_id STRING,
    total_orders BIGINT,
    delay_rate DOUBLE,
    average_review_score DOUBLE,
    spearman_correlation DOUBLE
)
USING DELTA
""")

# 9. GOLD_WEATHER_DELIVERY_DELAY
spark.sql("""
CREATE TABLE IF NOT EXISTS olist_delta.gold_weather_delivery_delay (
    order_id STRING,
    customer_state STRING,
    purchase_date DATE,
    delivery_date DATE,
    estimated_delivery_date DATE,
    days_late DOUBLE,
    accumulated_precipitation DOUBLE,
    weather_coverage DOUBLE
)
USING DELTA
""")

print("As 9 tabelas foram criadas!")

AnalysisException: Cannot create table ('`olist_delta`.`silver_orders`'). The associated location ('file:/home/jovyan/PL%2520EDSTD/spark-warehouse/olist_delta.db/silver_orders') is not empty and also not a Delta table.

In [4]:
import os
import shutil

caminho = "/home/jovyan/PL EDSTD/spark-warehouse/olist_delta.db/silver_orders"

if os.path.exists(caminho):
    shutil.rmtree(caminho)
    print("Pasta antiga silver_orders removida.")
else:
    print("A pasta não existe.")

A pasta não existe.


In [5]:
print("Localização do Warehouse:")
print(spark.conf.get("spark.sql.warehouse.dir"))

print("\nBase de dados olist_delta:")
spark.sql("DESCRIBE DATABASE EXTENDED olist_delta").show(truncate=False)

Localização do Warehouse:
file:/home/jovyan/PL%20EDSTD/spark-warehouse

Base de dados olist_delta:
+--------------+-------------------------------------------------------------+
|info_name     |info_value                                                   |
+--------------+-------------------------------------------------------------+
|Catalog Name  |spark_catalog                                                |
|Namespace Name|olist_delta                                                  |
|Comment       |                                                             |
|Location      |file:/home/jovyan/PL%2520EDSTD/spark-warehouse/olist_delta.db|
|Owner         |jovyan                                                       |
|Properties    |                                                             |
+--------------+-------------------------------------------------------------+



In [6]:
import os

# Criar uma pasta sem espaços para o armazenamento Delta
os.makedirs("/home/jovyan/spark-warehouse", exist_ok=True)

# Remover a base de dados antiga (está vazia)
spark.sql("DROP DATABASE IF EXISTS olist_delta CASCADE")

# Criar novamente a base de dados numa localização segura
spark.sql("""
CREATE DATABASE olist_delta
LOCATION 'file:/home/jovyan/spark-warehouse/olist_delta.db'
""")

print("Base de dados recriada com sucesso!")

Base de dados recriada com sucesso!


In [7]:
spark.sql("DESCRIBE DATABASE EXTENDED olist_delta").show(truncate=False)

+--------------+------------------------------------------------+
|info_name     |info_value                                      |
+--------------+------------------------------------------------+
|Catalog Name  |spark_catalog                                   |
|Namespace Name|olist_delta                                     |
|Comment       |                                                |
|Location      |file:/home/jovyan/spark-warehouse/olist_delta.db|
|Owner         |jovyan                                          |
|Properties    |                                                |
+--------------+------------------------------------------------+



In [8]:
# 1. SILVER_ORDERS
spark.sql("""
CREATE TABLE IF NOT EXISTS olist_delta.silver_orders (
    order_id STRING,
    customer_id STRING,
    order_status STRING,
    order_purchase_timestamp TIMESTAMP,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP
)
USING DELTA
""")

# 2. SILVER_ORDER_ITEMS
spark.sql("""
CREATE TABLE IF NOT EXISTS olist_delta.silver_order_items (
    order_id STRING,
    order_item_id INT,
    product_id STRING,
    seller_id STRING,
    shipping_limit_date TIMESTAMP,
    price DOUBLE,
    freight_value DOUBLE
)
USING DELTA
""")

# 3. SILVER_ORDER_REVIEWS
spark.sql("""
CREATE TABLE IF NOT EXISTS olist_delta.silver_order_reviews (
    review_id STRING,
    order_id STRING,
    review_score INT,
    review_comment_title STRING,
    review_comment_message STRING,
    review_creation_date TIMESTAMP,
    review_answer_timestamp TIMESTAMP
)
USING DELTA
""")

# 4. SILVER_SELLERS
spark.sql("""
CREATE TABLE IF NOT EXISTS olist_delta.silver_sellers (
    seller_id STRING,
    seller_zip_code_prefix STRING,
    seller_city STRING,
    seller_state STRING
)
USING DELTA
""")

# 5. SILVER_CUSTOMERS
spark.sql("""
CREATE TABLE IF NOT EXISTS olist_delta.silver_customers (
    customer_id STRING,
    customer_unique_id STRING,
    customer_zip_code_prefix STRING,
    customer_city STRING,
    customer_state STRING
)
USING DELTA
""")

# 6. SILVER_WEATHER_OBSERVATIONS
spark.sql("""
CREATE TABLE IF NOT EXISTS olist_delta.silver_weather_observations (
    station_id STRING,
    state_code STRING,
    observation_date DATE,
    observation_time STRING,
    precipitation DOUBLE,
    temperature DOUBLE,
    relative_humidity DOUBLE,
    latitude DOUBLE,
    longitude DOUBLE
)
USING DELTA
""")

# 7. SILVER_WEATHER_DAILY_STATE
spark.sql("""
CREATE TABLE IF NOT EXISTS olist_delta.silver_weather_daily_state (
    state_code STRING,
    observation_date DATE,
    avg_precipitation DOUBLE,
    avg_temperature DOUBLE,
    avg_relative_humidity DOUBLE,
    station_count INT,
    precipitation_coverage DOUBLE
)
USING DELTA
""")

# 8. GOLD_SELLER_PERFORMANCE
spark.sql("""
CREATE TABLE IF NOT EXISTS olist_delta.gold_seller_performance (
    seller_id STRING,
    total_orders BIGINT,
    delay_rate DOUBLE,
    average_review_score DOUBLE,
    spearman_correlation DOUBLE
)
USING DELTA
""")

# 9. GOLD_WEATHER_DELIVERY_DELAY
spark.sql("""
CREATE TABLE IF NOT EXISTS olist_delta.gold_weather_delivery_delay (
    order_id STRING,
    customer_state STRING,
    purchase_date DATE,
    delivery_date DATE,
    estimated_delivery_date DATE,
    days_late DOUBLE,
    accumulated_precipitation DOUBLE,
    weather_coverage DOUBLE
)
USING DELTA
""")

print("As 9 tabelas foram criadas!")

As 9 tabelas foram criadas!


In [9]:
spark.sql("SHOW TABLES IN olist_delta").show(truncate=False)

+-----------+---------------------------+-----------+
|namespace  |tableName                  |isTemporary|
+-----------+---------------------------+-----------+
|olist_delta|gold_seller_performance    |false      |
|olist_delta|gold_weather_delivery_delay|false      |
|olist_delta|silver_customers           |false      |
|olist_delta|silver_order_items         |false      |
|olist_delta|silver_order_reviews       |false      |
|olist_delta|silver_orders              |false      |
|olist_delta|silver_sellers             |false      |
|olist_delta|silver_weather_daily_state |false      |
|olist_delta|silver_weather_observations|false      |
+-----------+---------------------------+-----------+



In [10]:
import os

print("Pasta atual do Jupyter:")
print(os.getcwd())

print("\nFicheiros e pastas nesta localização:")
for item in os.listdir("."):
    print(item)

Pasta atual do Jupyter:
/home/jovyan/PL EDSTD

Ficheiros e pastas nesta localização:
Untitled.ipynb
sellers.html
olist_products_dataset.csv
orders.html
olist_orders_dataset.csv
olist_geolocation_dataset.csv
olist_sellers_dataset.csv
order_reviews.html
.ipynb_checkpoints
output_data_part2.csv
product_category_name_translation.csv
Untitled1.ipynb
order_payments.html
ydata.ipynb
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_customers_dataset.csv
olist_order_items_dataset.csv
products.html
order_items.html
output_data_part1.csv


In [11]:
import os

for pasta, subpastas, ficheiros in os.walk("."):
    subpastas[:] = [
        p for p in subpastas
        if p not in [".ipynb_checkpoints", "__pycache__"]
    ]

    for ficheiro in ficheiros:
        if ficheiro.lower().endswith(".csv"):
            print(os.path.join(pasta, ficheiro))

./olist_products_dataset.csv
./olist_orders_dataset.csv
./olist_geolocation_dataset.csv
./olist_sellers_dataset.csv
./output_data_part2.csv
./product_category_name_translation.csv
./olist_order_payments_dataset.csv
./olist_order_reviews_dataset.csv
./olist_customers_dataset.csv
./olist_order_items_dataset.csv
./output_data_part1.csv


In [12]:
import os

ficheiros = [
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_sellers_dataset.csv",
    "olist_customers_dataset.csv"
]

for ficheiro in ficheiros:
    caminho = os.path.join(os.getcwd(), ficheiro)
    print(f"{ficheiro}: {'Encontrado' if os.path.exists(caminho) else 'Não encontrado'}")

olist_orders_dataset.csv: Encontrado
olist_order_items_dataset.csv: Encontrado
olist_order_reviews_dataset.csv: Encontrado
olist_sellers_dataset.csv: Encontrado
olist_customers_dataset.csv: Encontrado


In [13]:
df_orders = spark.read.csv(
    "olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)

df_orders.printSchema()
df_orders.show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------

In [14]:
from pyspark.sql.functions import col

# 1. Extrair os dados do ficheiro CSV
df_orders = spark.read.csv(
    "olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)

# 2. Transformar os dados
# Remover registos duplicados pelo identificador da encomenda
df_orders = df_orders.dropDuplicates(["order_id"])

# Remover registos sem os identificadores essenciais
df_orders = df_orders.filter(
    col("order_id").isNotNull() &
    col("customer_id").isNotNull()
)

# 3. Verificar os dados após a transformação
print("Número de encomendas após limpeza:", df_orders.count())

df_orders.printSchema()
df_orders.show(5, truncate=False)

Número de encomendas após limpeza: 99441
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+--------------

In [15]:
# Guardar os dados transformados na tabela Silver
df_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_delta.silver_orders")

print("Tabela silver_orders carregada com sucesso!")

spark.sql("""
    SELECT COUNT(*) AS total_encomendas
    FROM olist_delta.silver_orders
""").show()

Tabela silver_orders carregada com sucesso!
+----------------+
|total_encomendas|
+----------------+
|           99441|
+----------------+



In [16]:
from pyspark.sql.functions import col

# 1. Extract: ler o ficheiro CSV
df_order_items = spark.read.csv(
    "olist_order_items_dataset.csv",
    header=True,
    inferSchema=True
)

# 2. Transform: remover duplicados e registos inválidos
df_order_items = df_order_items.dropDuplicates(
    ["order_id", "order_item_id"]
)

df_order_items = df_order_items.filter(
    col("order_id").isNotNull() &
    col("order_item_id").isNotNull() &
    col("seller_id").isNotNull() &
    col("price").isNotNull() &
    col("freight_value").isNotNull()
)

# Verificar os resultados
print("Número de itens após limpeza:", df_order_items.count())

df_order_items.printSchema()
df_order_items.show(5, truncate=False)

Número de itens após limpeza: 112650
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price|freight_value|
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+
|00018f77f2f0320c557190d7a144bdd3|1            |e5f2d52b802189ee658865ca93d83a8f|dd7ddc04e1b6c2c614352b383efe2d36|2017-05-03 11:05:13|239.9|19.93        |
|0015ebb40fb17286bea51d4607c4733

In [17]:
df_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_delta.silver_order_items")

print("Tabela silver_order_items carregada com sucesso!")

spark.sql("""
    SELECT COUNT(*) AS total_itens
    FROM olist_delta.silver_order_items
""").show()

Tabela silver_order_items carregada com sucesso!
+-----------+
|total_itens|
+-----------+
|     112650|
+-----------+



In [18]:
from pyspark.sql.functions import col

# 1. Extract: ler o ficheiro CSV
df_order_reviews = spark.read.csv(
    "olist_order_reviews_dataset.csv",
    header=True,
    inferSchema=True
)

# 2. Transform: remover duplicados
df_order_reviews = df_order_reviews.dropDuplicates(
    ["review_id", "order_id"]
)

# Remover registos sem identificadores ou classificação
df_order_reviews = df_order_reviews.filter(
    col("review_id").isNotNull() &
    col("order_id").isNotNull() &
    col("review_score").isNotNull()
)

# Verificar os resultados
print("Número de avaliações após limpeza:", df_order_reviews.count())

df_order_reviews.printSchema()
df_order_reviews.show(5, truncate=False)

Número de avaliações após limpeza: 101746
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)

+--------------------------------+--------------------------------+-------------------+--------------------+-----------------------------------------------------+--------------------+-----------------------+
|review_id                       |order_id                        |review_score       |review_comment_title|review_comment_message                               |review_creation_date|review_answer_timestamp|
+--------------------------------+--------------------------------+-------------------+--------------------+-----------------------------------------------------+--------------------+---------------

In [19]:
df_order_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_delta.silver_order_reviews")

print("Tabela silver_order_reviews carregada com sucesso!")

spark.sql("""
    SELECT COUNT(*) AS total_avaliacoes
    FROM olist_delta.silver_order_reviews
""").show()

AnalysisException: Failed to merge fields 'review_score' and 'review_score'. Failed to merge incompatible data types IntegerType and StringType

In [20]:
from pyspark.sql.functions import col

df_order_reviews = df_order_reviews.withColumn(
    "review_score",
    col("review_score").cast("int")
)

df_order_reviews.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)



In [21]:
df_order_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_delta.silver_order_reviews")

print("Tabela silver_order_reviews carregada com sucesso!")

spark.sql("""
    SELECT COUNT(*) AS total_avaliacoes
    FROM olist_delta.silver_order_reviews
""").show()

AnalysisException: Failed to merge fields 'review_creation_date' and 'review_creation_date'. Failed to merge incompatible data types TimestampType and StringType

In [22]:
from pyspark.sql.functions import col, to_timestamp

df_order_reviews = df_order_reviews.withColumn(
    "review_creation_date",
    to_timestamp(col("review_creation_date"), "yyyy-MM-dd HH:mm:ss")
)

df_order_reviews = df_order_reviews.withColumn(
    "review_answer_timestamp",
    to_timestamp(col("review_answer_timestamp"), "yyyy-MM-dd HH:mm:ss")
)

df_order_reviews.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)



In [23]:
df_order_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_delta.silver_order_reviews")

print("Tabela silver_order_reviews carregada com sucesso!")

spark.sql("""
    SELECT COUNT(*) AS total_avaliacoes
    FROM olist_delta.silver_order_reviews
""").show()

Tabela silver_order_reviews carregada com sucesso!
+----------------+
|total_avaliacoes|
+----------------+
|          101746|
+----------------+



In [24]:
from pyspark.sql.functions import col

# 1. Extrair os dados do ficheiro CSV
df_sellers = spark.read.csv(
    "olist_sellers_dataset.csv",
    header=True,
    inferSchema=True
)

# 2. Transformar os dados
# Remover vendedores duplicados
df_sellers = df_sellers.dropDuplicates(["seller_id"])

# Remover registos sem identificador
df_sellers = df_sellers.filter(
    col("seller_id").isNotNull()
)

# 3. Verificar os dados
print("Número de vendedores após limpeza:", df_sellers.count())

df_sellers.printSchema()
df_sellers.show(5, truncate=False)

Número de vendedores após limpeza: 3095
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)

+--------------------------------+----------------------+-----------+------------+
|seller_id                       |seller_zip_code_prefix|seller_city|seller_state|
+--------------------------------+----------------------+-----------+------------+
|0015a82c2db000af6aaaf3ae2ecb0532|9080                  |santo andre|SP          |
|001cca7ae9ae17fb1caed9dfb1094831|29156                 |cariacica  |ES          |
|001e6ad469a905060d959994f1b41e4f|24754                 |sao goncalo|RJ          |
|002100f778ceb8431b7a1020ff7ab48f|14405                 |franca     |SP          |
|003554e2dce176b5555353e4f3555ac8|74565                 |goiania    |GO          |
+--------------------------------+----------------------+-----------+------------+
only showing top 5 rows


In [25]:
df_sellers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_delta.silver_sellers")

print("Tabela silver_sellers carregada com sucesso!")

spark.sql("""
    SELECT COUNT(*) AS total_vendedores
    FROM olist_delta.silver_sellers
""").show()

AnalysisException: Failed to merge fields 'seller_zip_code_prefix' and 'seller_zip_code_prefix'. Failed to merge incompatible data types StringType and IntegerType

In [26]:
from pyspark.sql.functions import col

df_sellers = df_sellers.withColumn(
    "seller_zip_code_prefix",
    col("seller_zip_code_prefix").cast("int")
)

df_sellers.printSchema()

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



In [27]:
df_sellers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_delta.silver_sellers")

print("Tabela silver_sellers carregada com sucesso!")

AnalysisException: Failed to merge fields 'seller_zip_code_prefix' and 'seller_zip_code_prefix'. Failed to merge incompatible data types StringType and IntegerType

In [28]:
print("ESQUEMA DO DATAFRAME:")
df_sellers.printSchema()

print("ESQUEMA DA TABELA DELTA:")
spark.sql("DESCRIBE TABLE olist_delta.silver_sellers").show(truncate=False)

ESQUEMA DO DATAFRAME:
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)

ESQUEMA DA TABELA DELTA:
+----------------------+---------+-------+
|col_name              |data_type|comment|
+----------------------+---------+-------+
|seller_id             |string   |null   |
|seller_zip_code_prefix|string   |null   |
|seller_city           |string   |null   |
|seller_state          |string   |null   |
+----------------------+---------+-------+



In [29]:
df_sellers = df_sellers.withColumn(
    "seller_zip_code_prefix",
    col("seller_zip_code_prefix").cast("string")
)

df_sellers.printSchema()

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



In [30]:
df_sellers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_delta.silver_sellers")

print("Tabela silver_sellers carregada com sucesso!")

Tabela silver_sellers carregada com sucesso!


In [31]:
spark.sql("""
    SELECT COUNT(*) AS total_vendedores
    FROM olist_delta.silver_sellers
""").show()

spark.sql("""
    SELECT *
    FROM olist_delta.silver_sellers
    LIMIT 5
""").show(truncate=False)

+----------------+
|total_vendedores|
+----------------+
|            3095|
+----------------+

+--------------------------------+----------------------+-----------+------------+
|seller_id                       |seller_zip_code_prefix|seller_city|seller_state|
+--------------------------------+----------------------+-----------+------------+
|0015a82c2db000af6aaaf3ae2ecb0532|9080                  |santo andre|SP          |
|001cca7ae9ae17fb1caed9dfb1094831|29156                 |cariacica  |ES          |
|001e6ad469a905060d959994f1b41e4f|24754                 |sao goncalo|RJ          |
|002100f778ceb8431b7a1020ff7ab48f|14405                 |franca     |SP          |
|003554e2dce176b5555353e4f3555ac8|74565                 |goiania    |GO          |
+--------------------------------+----------------------+-----------+------------+



In [32]:
from pyspark.sql.functions import col

# 1. Extrair os dados do ficheiro CSV
df_customers = spark.read.csv(
    "olist_customers_dataset.csv",
    header=True,
    inferSchema=True
)

# 2. Remover clientes duplicados
df_customers = df_customers.dropDuplicates(["customer_id"])

# 3. Remover registos sem identificador
df_customers = df_customers.filter(
    col("customer_id").isNotNull()
)

# 4. Verificar os dados
print("Número de clientes após limpeza:", df_customers.count())

df_customers.printSchema()
df_customers.show(5, truncate=False)

Número de clientes após limpeza: 99441
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

+--------------------------------+--------------------------------+------------------------+-------------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city      |customer_state|
+--------------------------------+--------------------------------+------------------------+-------------------+--------------+
|00050bf6e01e69d5c0fd612f1bcfb69c|e3cf594a99e810f58af53ed4820f25e5|98700                   |ijui               |RS            |
|000598caf2ef4117407665ac33275130|7e0516b486e92ed3f3afdd6d1276cfbd|35540                   |oliveira           |MG            |
|0013cd8e350a7cc76873441e431dd5ee|334fed5abcee3aa96c13f1432703e1fd|3585   

In [33]:
df_customers = df_customers.withColumn(
    "customer_zip_code_prefix",
    col("customer_zip_code_prefix").cast("string")
)

df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [34]:
df_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_delta.silver_customers")

print("Tabela silver_customers carregada com sucesso!")

Tabela silver_customers carregada com sucesso!


In [35]:
df_weather = spark.read.csv(
    "output_data_part1.csv",
    header=True,
    inferSchema=True
)

print("ESQUEMA:")
df_weather.printSchema()

print("PRIMEIROS REGISTOS:")
df_weather.show(5, truncate=False)

ESQUEMA:
root
 |-- CodigoOMM;NomeEstacao;Estado;EstadoDesc;EstadoRegiao;Latitude;Longitude;Altitude;EstacaoStituacao;OperanteDesde;Estacao;Data;Hora;Precipitacao;TempBulboSeco;TempBulboUmido;TempMaxima;TempMinima;UmidadeRelativa;PressaoAtmEstacao;PressaoAtmMar;DirecaoVento;VelocidadeVentoInsolacao;Nebulosidade;Evaporacao Piche;Temp Comp Media;Umidade Relativa Media;Velocidade do Vento Media: string (nullable = true)

PRIMEIROS REGISTOS:
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|CodigoOMM;NomeEstacao;Estado;EstadoDesc;EstadoRegiao;Latitude;Longitude;Altitude;EstacaoStituacao;OperanteDesde;Estacao;Data;Hora;Precipitacao;TempBulboSeco;TempBulboUmido;TempMaxim

In [36]:
df_weather = spark.read.csv(
    "output_data_part1.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

print("ESQUEMA:")
df_weather.printSchema()

print("PRIMEIROS REGISTOS:")
df_weather.show(5, truncate=False)

ESQUEMA:
root
 |-- CodigoOMM: integer (nullable = true)
 |-- NomeEstacao: string (nullable = true)
 |-- Estado: string (nullable = true)
 |-- EstadoDesc: string (nullable = true)
 |-- EstadoRegiao: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Altitude: double (nullable = true)
 |-- EstacaoStituacao: string (nullable = true)
 |-- OperanteDesde: string (nullable = true)
 |-- Estacao: integer (nullable = true)
 |-- Data: string (nullable = true)
 |-- Hora: integer (nullable = true)
 |-- Precipitacao: double (nullable = true)
 |-- TempBulboSeco: double (nullable = true)
 |-- TempBulboUmido: double (nullable = true)
 |-- TempMaxima: double (nullable = true)
 |-- TempMinima: double (nullable = true)
 |-- UmidadeRelativa: integer (nullable = true)
 |-- PressaoAtmEstacao: double (nullable = true)
 |-- PressaoAtmMar: double (nullable = true)
 |-- DirecaoVento: double (nullable = true)
 |-- VelocidadeVentoInsolacao: double (nullab

In [37]:
df_weather2 = spark.read.csv(
    "output_data_part2.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

print("ESQUEMA DO SEGUNDO FICHEIRO:")
df_weather2.printSchema()

print("PRIMEIROS REGISTOS:")
df_weather2.show(5, truncate=False)

ESQUEMA DO SEGUNDO FICHEIRO:
root
 |-- CodigoOMM: integer (nullable = true)
 |-- NomeEstacao: string (nullable = true)
 |-- Estado: string (nullable = true)
 |-- EstadoDesc: string (nullable = true)
 |-- EstadoRegiao: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Altitude: double (nullable = true)
 |-- EstacaoStituacao: string (nullable = true)
 |-- OperanteDesde: string (nullable = true)
 |-- Estacao: integer (nullable = true)
 |-- Data: string (nullable = true)
 |-- Hora: integer (nullable = true)
 |-- Precipitacao: double (nullable = true)
 |-- TempBulboSeco: double (nullable = true)
 |-- TempBulboUmido: double (nullable = true)
 |-- TempMaxima: double (nullable = true)
 |-- TempMinima: double (nullable = true)
 |-- UmidadeRelativa: integer (nullable = true)
 |-- PressaoAtmEstacao: double (nullable = true)
 |-- PressaoAtmMar: double (nullable = true)
 |-- DirecaoVento: double (nullable = true)
 |-- VelocidadeVentoInsol

In [38]:
from pyspark.sql.functions import col, to_date

# 1. Juntar os dois ficheiros
df_weather = df_weather.unionByName(df_weather2)

# 2. Selecionar apenas as colunas necessárias
df_weather = df_weather.select(
    col("CodigoOMM").cast("string").alias("station_id"),
    col("Estado").alias("state_code"),
    to_date(col("Data"), "dd/MM/yyyy").alias("observation_date"),
    col("Hora").cast("string").alias("observation_time"),
    col("Precipitacao").cast("double").alias("precipitation"),
    col("TempBulboSeco").cast("double").alias("temperature"),
    col("UmidadeRelativa").cast("double").alias("relative_humidity"),
    col("Latitude").cast("double").alias("latitude"),
    col("Longitude").cast("double").alias("longitude")
)

# 3. Verificar os dados
print("Total de observações:", df_weather.count())

df_weather.printSchema()
df_weather.show(5, truncate=False)

Total de observações: 1845665
root
 |-- station_id: string (nullable = true)
 |-- state_code: string (nullable = true)
 |-- observation_date: date (nullable = true)
 |-- observation_time: string (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- relative_humidity: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)

+----------+----------+----------------+----------------+-------------+-----------+-----------------+--------+---------+
|station_id|state_code|observation_date|observation_time|precipitation|temperature|relative_humidity|latitude|longitude|
+----------+----------+----------------+----------------+-------------+-----------+-----------------+--------+---------+
|82989     |AL        |2012-01-01      |0               |null         |25.1       |91.0             |-9.28   |-37.9    |
|82989     |AL        |2012-01-01      |1200            |0.0          |26.0       |8

In [39]:
# 1. Remover registos duplicados
df_weather = df_weather.dropDuplicates(
    ["station_id", "observation_date", "observation_time"]
)

# 2. Remover registos sem informação essencial
df_weather = df_weather.filter(
    col("station_id").isNotNull() &
    col("state_code").isNotNull() &
    col("observation_date").isNotNull()
)

# 3. Verificar os dados após limpeza
print("Total de observações após limpeza:", df_weather.count())

df_weather.show(5, truncate=False)

Total de observações após limpeza: 1845665
+----------+----------+----------------+----------------+-------------+-----------+-----------------+--------+---------+
|station_id|state_code|observation_date|observation_time|precipitation|temperature|relative_humidity|latitude|longitude|
+----------+----------+----------------+----------------+-------------+-----------+-----------------+--------+---------+
|82024     |RR        |2012-01-04      |1200            |0.0          |27.5       |72.0             |2.82    |-60.66   |
|82024     |RR        |2012-01-06      |1800            |null         |32.5       |55.0             |2.82    |-60.66   |
|82024     |RR        |2012-01-13      |0               |null         |29.9       |64.0             |2.82    |-60.66   |
|82024     |RR        |2012-02-01      |1200            |0.0          |25.8       |80.0             |2.82    |-60.66   |
|82024     |RR        |2012-02-08      |1200            |0.0          |24.8       |91.0             |2.82    |

In [40]:
df_weather.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_delta.silver_weather_observations")

print("Tabela silver_weather_observations carregada com sucesso!")

Tabela silver_weather_observations carregada com sucesso!


In [41]:
spark.sql("""
    SELECT COUNT(*) AS total_observacoes
    FROM olist_delta.silver_weather_observations
""").show()

+-----------------+
|total_observacoes|
+-----------------+
|          1845665|
+-----------------+



In [42]:
df_weather.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_delta.silver_weather_observations")

print("Tabela silver_weather_observations carregada com sucesso!")

Tabela silver_weather_observations carregada com sucesso!


In [43]:
from pyspark.sql.functions import (
    col, avg, count, countDistinct, 
    when, round
)

df_weather_daily = (
    df_weather
    .groupBy("state_code", "observation_date")
    .agg(
        avg("precipitation").alias("avg_precipitation"),
        avg("temperature").alias("avg_temperature"),
        avg("relative_humidity").alias("avg_relative_humidity"),
        countDistinct("station_id").alias("station_count"),
        round(
            count("precipitation") / count("*"),
            4
        ).alias("precipitation_coverage")
    )
)

print("Total de registos por estado e dia:", df_weather_daily.count())

df_weather_daily.printSchema()
df_weather_daily.show(10, truncate=False)

Total de registos por estado e dia: 74418
root
 |-- state_code: string (nullable = true)
 |-- observation_date: date (nullable = true)
 |-- avg_precipitation: double (nullable = true)
 |-- avg_temperature: double (nullable = true)
 |-- avg_relative_humidity: double (nullable = true)
 |-- station_count: long (nullable = false)
 |-- precipitation_coverage: double (nullable = true)

+----------+----------------+------------------+------------------+---------------------+-------------+----------------------+
|state_code|observation_date|avg_precipitation |avg_temperature   |avg_relative_humidity|station_count|precipitation_coverage|
+----------+----------------+------------------+------------------+---------------------+-------------+----------------------+
|BA        |2014-02-07      |1.67037037037037  |25.817333333333337|72.28                |27           |0.3506                |
|MG        |2013-01-13      |8.225             |23.520833333333332|76.76666666666667    |42           |0.3252

In [44]:
df_weather_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_delta.silver_weather_daily_state")

print("Tabela silver_weather_daily_state carregada com sucesso!")

AnalysisException: Failed to merge fields 'station_count' and 'station_count'. Failed to merge incompatible data types IntegerType and LongType

In [45]:
df_weather_daily = df_weather_daily.withColumn(
    "station_count",
    col("station_count").cast("int")
)

df_weather_daily.printSchema()

root
 |-- state_code: string (nullable = true)
 |-- observation_date: date (nullable = true)
 |-- avg_precipitation: double (nullable = true)
 |-- avg_temperature: double (nullable = true)
 |-- avg_relative_humidity: double (nullable = true)
 |-- station_count: integer (nullable = false)
 |-- precipitation_coverage: double (nullable = true)



In [46]:
df_weather_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_delta.silver_weather_daily_state")

print("Tabela silver_weather_daily_state carregada com sucesso!")

Tabela silver_weather_daily_state carregada com sucesso!


In [47]:
spark.sql("""
    SELECT COUNT(*) AS total_registos
    FROM olist_delta.silver_weather_daily_state
""").show()

+--------------+
|total_registos|
+--------------+
|         74418|
+--------------+



In [48]:
spark.sql("SHOW TABLES IN olist_delta").show(truncate=False)

+-----------+---------------------------+-----------+
|namespace  |tableName                  |isTemporary|
+-----------+---------------------------+-----------+
|olist_delta|gold_seller_performance    |false      |
|olist_delta|gold_weather_delivery_delay|false      |
|olist_delta|silver_customers           |false      |
|olist_delta|silver_order_items         |false      |
|olist_delta|silver_order_reviews       |false      |
|olist_delta|silver_orders              |false      |
|olist_delta|silver_sellers             |false      |
|olist_delta|silver_weather_daily_state |false      |
|olist_delta|silver_weather_observations|false      |
+-----------+---------------------------+-----------+



In [49]:
tabelas = [
    "silver_orders",
    "silver_order_items",
    "silver_order_reviews",
    "silver_sellers",
    "silver_customers",
    "silver_weather_observations",
    "silver_weather_daily_state"
]

for tabela in tabelas:
    total = spark.table("olist_delta." + tabela).count()
    print(tabela, ":", total, "registos")

silver_orders : 99441 registos
silver_order_items : 112650 registos
silver_order_reviews : 101746 registos
silver_sellers : 3095 registos
silver_customers : 99441 registos
silver_weather_observations : 1845665 registos
silver_weather_daily_state : 74418 registos


In [50]:
from pyspark.sql.functions import sum, when

for tabela in [
    "silver_orders",
    "silver_order_items",
    "silver_order_reviews",
    "silver_sellers",
    "silver_customers",
    "silver_weather_observations",
    "silver_weather_daily_state"
]:

    df = spark.table("olist_delta." + tabela)

    print("\n---", tabela, "---")

    resultado = df.select([
        sum(
            when(col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in df.columns
    ])

    resultado.show(truncate=False)


--- silver_orders ---
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|0       |0          |0           |0                       |160              |1783                        |2965                         |0                            |
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+


--- silver_order_items ---
+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_i

In [51]:
from pyspark.sql.functions import col

orders = spark.table("olist_delta.silver_orders")
items = spark.table("olist_delta.silver_order_items")
reviews = spark.table("olist_delta.silver_order_reviews")
sellers = spark.table("olist_delta.silver_sellers")
customers = spark.table("olist_delta.silver_customers")

# Artigos com order_id inexistente na tabela de encomendas
items_sem_encomenda = (
    items.join(
        orders.select("order_id"),
        "order_id",
        "left_anti"
    ).count()
)

# Artigos com seller_id inexistente na tabela de vendedores
items_sem_vendedor = (
    items.join(
        sellers.select("seller_id"),
        "seller_id",
        "left_anti"
    ).count()
)

# Avaliações com order_id inexistente na tabela de encomendas
reviews_sem_encomenda = (
    reviews.join(
        orders.select("order_id"),
        "order_id",
        "left_anti"
    ).count()
)

# Encomendas com customer_id inexistente na tabela de clientes
orders_sem_cliente = (
    orders.join(
        customers.select("customer_id"),
        "customer_id",
        "left_anti"
    ).count()
)

print("Artigos sem encomenda correspondente:", items_sem_encomenda)
print("Artigos sem vendedor correspondente:", items_sem_vendedor)
print("Avaliações sem encomenda correspondente:", reviews_sem_encomenda)
print("Encomendas sem cliente correspondente:", orders_sem_cliente)

Artigos sem encomenda correspondente: 0
Artigos sem vendedor correspondente: 0
Avaliações sem encomenda correspondente: 2522
Encomendas sem cliente correspondente: 0


In [52]:
def verificar_duplicados(df, colunas, nome):
    total = df.count()

    distintos = df.select(*colunas).dropDuplicates().count()

    duplicados = total - distintos

    print(nome, ":", duplicados, "registos duplicados")


verificar_duplicados(
    orders,
    ["order_id"],
    "silver_orders"
)

verificar_duplicados(
    items,
    ["order_id", "order_item_id"],
    "silver_order_items"
)

verificar_duplicados(
    reviews,
    ["review_id"],
    "silver_order_reviews"
)

verificar_duplicados(
    sellers,
    ["seller_id"],
    "silver_sellers"
)

verificar_duplicados(
    customers,
    ["customer_id"],
    "silver_customers"
)

verificar_duplicados(
    spark.table("olist_delta.silver_weather_observations"),
    ["station_id", "observation_date", "observation_time"],
    "silver_weather_observations"
)

verificar_duplicados(
    spark.table("olist_delta.silver_weather_daily_state"),
    ["state_code", "observation_date"],
    "silver_weather_daily_state"
)

silver_orders : 0 registos duplicados
silver_order_items : 0 registos duplicados
silver_order_reviews : 1097 registos duplicados
silver_sellers : 0 registos duplicados
silver_customers : 0 registos duplicados
silver_weather_observations : 0 registos duplicados
silver_weather_daily_state : 0 registos duplicados


In [53]:
customers = spark.table("olist_delta.silver_customers")
weather = spark.table("olist_delta.silver_weather_daily_state")

estados_clientes = (
    customers
    .select("customer_state")
    .distinct()
)

estados_meteorologia = (
    weather
    .select(col("state_code").alias("customer_state"))
    .distinct()
)

estados_sem_meteorologia = (
    estados_clientes
    .join(
        estados_meteorologia,
        "customer_state",
        "left_anti"
    )
)

print("Estados dos clientes:")
estados_clientes.orderBy("customer_state").show()

print("Estados sem correspondência meteorológica:")
estados_sem_meteorologia.show()

print(
    "Total de estados sem correspondência:",
    estados_sem_meteorologia.count()
)

Estados dos clientes:
+--------------+
|customer_state|
+--------------+
|            AC|
|            AL|
|            AM|
|            AP|
|            BA|
|            CE|
|            DF|
|            ES|
|            GO|
|            MA|
|            MG|
|            MS|
|            MT|
|            PA|
|            PB|
|            PE|
|            PI|
|            PR|
|            RJ|
|            RN|
+--------------+
only showing top 20 rows

Estados sem correspondência meteorológica:
+--------------+
|customer_state|
+--------------+
|            RO|
+--------------+

Total de estados sem correspondência: 1


In [54]:
weather_daily = spark.table("olist_delta.silver_weather_daily_state")

weather_daily.select(
    "precipitation_coverage"
).summary().show()

+-------+----------------------+
|summary|precipitation_coverage|
+-------+----------------------+
|  count|                 74418|
|   mean|   0.35288925260018367|
| stddev|  0.042478261394607635|
|    min|                   0.0|
|    25%|                0.3333|
|    50%|                0.3415|
|    75%|                0.3667|
|    max|                   1.0|
+-------+----------------------+



In [55]:
print(
    "Registos sem precipitação média:",
    weather_daily.filter(
        col("avg_precipitation").isNull()
    ).count()
)

Registos sem precipitação média: 135


In [56]:
from pyspark.sql.functions import col

df_weather = spark.table("olist_delta.silver_weather_observations")

(
    df_weather
    .select("state_code")
    .distinct()
    .orderBy("state_code")
    .show(50, truncate=False)
)

+----------+
|state_code|
+----------+
|AC        |
|AL        |
|AM        |
|AP        |
|BA        |
|CE        |
|DF        |
|ES        |
|GO        |
|MA        |
|MG        |
|MS        |
|MT        |
|PA        |
|PB        |
|PE        |
|PI        |
|PR        |
|RJ        |
|RN        |
|RR        |
|RS        |
|SC        |
|SE        |
|SP        |
|TO        |
+----------+



In [57]:
from pyspark.sql.functions import upper, trim

(
    df_weather
    .select(upper(trim(col("state_code"))).alias("estado"))
    .distinct()
    .filter(
        col("estado").contains("RO")
    )
    .show(50, truncate=False)
)

+------+
|estado|
+------+
+------+



In [63]:
# Lista das tabelas Silver
tabelas_silver = [
    "silver_orders",
    "silver_order_items",
    "silver_order_reviews",
    "silver_sellers",
    "silver_customers",
    "silver_weather_observations",
    "silver_weather_daily_state"
]

# Número máximo de linhas a apresentar
limite = 100

# Visualizar cada tabela
for tabela in tabelas_silver:

    df = spark.table("olist_delta." + tabela)

    print(f"\n{tabela}")
    print(f"Total de registos: {df.count():,}")
    print(f"Total de colunas: {len(df.columns)}")

    display(df.limit(limite))


silver_orders
Total de registos: 99,441
Total de colunas: 8


DataFrame[order_id: string, customer_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp]


silver_order_items
Total de registos: 112,650
Total de colunas: 7


DataFrame[order_id: string, order_item_id: int, product_id: string, seller_id: string, shipping_limit_date: timestamp, price: double, freight_value: double]


silver_order_reviews
Total de registos: 101,746
Total de colunas: 7


DataFrame[review_id: string, order_id: string, review_score: int, review_comment_title: string, review_comment_message: string, review_creation_date: timestamp, review_answer_timestamp: timestamp]

... (4 outputs left out)


silver_weather_observations
Total de registos: 1,845,665
Total de colunas: 9


DataFrame[station_id: string, state_code: string, observation_date: date, observation_time: string, precipitation: double, temperature: double, relative_humidity: double, latitude: double, longitude: double]


silver_weather_daily_state
Total de registos: 74,418
Total de colunas: 7


DataFrame[state_code: string, observation_date: date, avg_precipitation: double, avg_temperature: double, avg_relative_humidity: double, station_count: int, precipitation_coverage: double]

In [1]:

# 1. Listar todas as tabelas Silver
spark.sql("SHOW TABLES IN olist_delta").show(truncate=False)


# 2. Definir as tabelas Silver
tabelas_silver = [
    "silver_orders",
    "silver_order_items",
    "silver_order_reviews",
    "silver_sellers",
    "silver_customers",
    "silver_weather_observations",
    "silver_weather_daily_state"
]


# 3. Visualizar cada tabela
for tabela in tabelas_silver:

    print("\n" + "=" * 80)
    print("TABELA:", tabela)
    print("=" * 80)

    df = spark.table("olist_delta." + tabela)

    # Número de registos
    print("Total de registos:", df.count())

    # Estrutura da tabela
    df.printSchema()

    # Mostrar os primeiros 20 registos
    df.show(20, truncate=False)


NameError: name 'spark' is not defined

In [2]:

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

builder = (
    SparkSession.builder
    .appName("Olist")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark iniciado com sucesso!")


Spark iniciado com sucesso!


In [3]:

spark.sql("SHOW DATABASES").show(truncate=False)


+---------+
|namespace|
+---------+
|default  |
+---------+



In [4]:
spark.sql("SHOW TABLES IN olist_delta").show(truncate=False)


AnalysisException: [SCHEMA_NOT_FOUND] The schema `olist_delta` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a catalog, verify the current_schema() output, or qualify the name with the correct catalog.
To tolerate the error on drop use DROP SCHEMA IF EXISTS.

In [5]:

# 1. Listar todas as tabelas Silver
spark.sql("SHOW TABLES IN olist_delta").show(truncate=False)


# 2. Definir as tabelas Silver
tabelas_silver = [
    "silver_orders",
    "silver_order_items",
    "silver_order_reviews",
    "silver_sellers",
    "silver_customers",
    "silver_weather_observations",
    "silver_weather_daily_state"
]


# 3. Visualizar cada tabela
for tabela in tabelas_silver:

    print("\n" + "=" * 80)
    print("TABELA:", tabela)
    print("=" * 80)

    df = spark.table("olist_delta." + tabela)

    # Número de registos
    print("Total de registos:", df.count())

    # Estrutura da tabela
    df.printSchema()

    # Mostrar os primeiros 20 registos
    df.show(20, truncate=False)


AnalysisException: [SCHEMA_NOT_FOUND] The schema `olist_delta` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a catalog, verify the current_schema() output, or qualify the name with the correct catalog.
To tolerate the error on drop use DROP SCHEMA IF EXISTS.

In [6]:

import os

# Verificar a base de dados atual
print("Base de dados atual:")
spark.sql("SELECT current_database()").show()

# Verificar as bases de dados disponíveis
print("Bases de dados disponíveis:")
spark.sql("SHOW DATABASES").show(truncate=False)

# Verificar a localização do warehouse
print("Warehouse atual:")
print(spark.conf.get("spark.sql.warehouse.dir"))

# Verificar se a pasta antiga ainda existe
caminho = "/home/jovyan/spark-warehouse/olist_delta.db"

print("\nA pasta antiga existe?")
print(os.path.exists(caminho))

if os.path.exists(caminho):
    print("\nConteúdo da pasta:")
    print(os.listdir(caminho))


Base de dados atual:
+------------------+
|current_database()|
+------------------+
|           default|
+------------------+

Bases de dados disponíveis:
+---------+
|namespace|
+---------+
|default  |
+---------+

Warehouse atual:
file:/home/jovyan/PL%20EDSTD/spark-warehouse

A pasta antiga existe?
True

Conteúdo da pasta:
['silver_sellers', 'silver_weather_observations', 'silver_order_items', 'silver_orders', 'silver_weather_daily_state', 'gold_seller_performance', 'gold_weather_delivery_delay', 'silver_order_reviews', 'silver_customers']


In [7]:

spark.sql("""
CREATE DATABASE IF NOT EXISTS olist_delta
LOCATION 'file:/home/jovyan/spark-warehouse/olist_delta.db'
""")

print("Base de dados olist_delta recuperada!")


Base de dados olist_delta recuperada!


In [8]:

spark.sql("SHOW TABLES IN olist_delta").show(truncate=False)


+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [9]:

import os

base = "/home/jovyan/spark-warehouse/olist_delta.db"

tabelas = [
    "silver_orders",
    "silver_order_items",
    "silver_order_reviews",
    "silver_sellers",
    "silver_customers",
    "silver_weather_observations",
    "silver_weather_daily_state",
    "gold_seller_performance",
    "gold_weather_delivery_delay"
]

for tabela in tabelas:
    caminho_log = os.path.join(base, tabela, "_delta_log")

    print(
        tabela,
        "->",
        "Delta encontrado" if os.path.exists(caminho_log) else "SEM DELTA LOG"
    )


silver_orders -> Delta encontrado
silver_order_items -> Delta encontrado
silver_order_reviews -> Delta encontrado
silver_sellers -> Delta encontrado
silver_customers -> Delta encontrado
silver_weather_observations -> Delta encontrado
silver_weather_daily_state -> Delta encontrado
gold_seller_performance -> Delta encontrado
gold_weather_delivery_delay -> Delta encontrado


In [10]:

for tabela in tabelas:

    caminho = f"{base}/{tabela}"

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS olist_delta.{tabela}
        USING DELTA
        LOCATION '{caminho}'
    """)

    print("Tabela registada:", tabela)


Tabela registada: silver_orders
Tabela registada: silver_order_items
Tabela registada: silver_order_reviews
Tabela registada: silver_sellers
Tabela registada: silver_customers
Tabela registada: silver_weather_observations
Tabela registada: silver_weather_daily_state
Tabela registada: gold_seller_performance
Tabela registada: gold_weather_delivery_delay


In [11]:

spark.sql("SHOW TABLES IN olist_delta").show(truncate=False)

print("Total de encomendas:")
spark.table("olist_delta.silver_orders").count()


+-----------+---------------------------+-----------+
|namespace  |tableName                  |isTemporary|
+-----------+---------------------------+-----------+
|olist_delta|gold_seller_performance    |false      |
|olist_delta|gold_weather_delivery_delay|false      |
|olist_delta|silver_customers           |false      |
|olist_delta|silver_order_items         |false      |
|olist_delta|silver_order_reviews       |false      |
|olist_delta|silver_orders              |false      |
|olist_delta|silver_sellers             |false      |
|olist_delta|silver_weather_daily_state |false      |
|olist_delta|silver_weather_observations|false      |
+-----------+---------------------------+-----------+

Total de encomendas:


99441

In [12]:

# Tabelas Silver do projeto Olist
tabelas_silver = [
    "silver_orders",
    "silver_order_items",
    "silver_order_reviews",
    "silver_sellers",
    "silver_customers",
    "silver_weather_observations",
    "silver_weather_daily_state"
]

# Percorrer todas as tabelas
for tabela in tabelas_silver:

    df = spark.table("olist_delta." + tabela)

    print("\n" + "=" * 100)
    print("TABELA:", tabela)
    print("=" * 100)

    # Número de registos
    print("Número de registos:", df.count())

    # Estrutura e tipos de dados
    print("\nESTRUTURA:")
    df.printSchema()

    # Visualização dos primeiros 20 registos
    print("\nPRIMEIROS 20 REGISTOS:")
    display(df.limit(20))



TABELA: silver_orders
Número de registos: 99441

ESTRUTURA:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)


PRIMEIROS 20 REGISTOS:


DataFrame[order_id: string, customer_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp]


TABELA: silver_order_items
Número de registos: 112650

ESTRUTURA:
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)


PRIMEIROS 20 REGISTOS:


DataFrame[order_id: string, order_item_id: int, product_id: string, seller_id: string, shipping_limit_date: timestamp, price: double, freight_value: double]


TABELA: silver_order_reviews
Número de registos: 101746

ESTRUTURA:
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)


PRIMEIROS 20 REGISTOS:


DataFrame[review_id: string, order_id: string, review_score: int, review_comment_title: string, review_comment_message: string, review_creation_date: timestamp, review_answer_timestamp: timestamp]

... (4 outputs left out)


TABELA: silver_weather_observations
Número de registos: 1845665

ESTRUTURA:
root
 |-- station_id: string (nullable = true)
 |-- state_code: string (nullable = true)
 |-- observation_date: date (nullable = true)
 |-- observation_time: string (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- relative_humidity: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)


PRIMEIROS 20 REGISTOS:


DataFrame[station_id: string, state_code: string, observation_date: date, observation_time: string, precipitation: double, temperature: double, relative_humidity: double, latitude: double, longitude: double]


TABELA: silver_weather_daily_state
Número de registos: 74418

ESTRUTURA:
root
 |-- state_code: string (nullable = true)
 |-- observation_date: date (nullable = true)
 |-- avg_precipitation: double (nullable = true)
 |-- avg_temperature: double (nullable = true)
 |-- avg_relative_humidity: double (nullable = true)
 |-- station_count: integer (nullable = true)
 |-- precipitation_coverage: double (nullable = true)


PRIMEIROS 20 REGISTOS:


DataFrame[state_code: string, observation_date: date, avg_precipitation: double, avg_temperature: double, avg_relative_humidity: double, station_count: int, precipitation_coverage: double]

In [13]:
# Lista das tabelas Silver
tabelas_silver = [
    "silver_orders",
    "silver_order_items",
    "silver_order_reviews",
    "silver_sellers",
    "silver_customers",
    "silver_weather_observations",
    "silver_weather_daily_state"
]

# Visualizar os primeiros 5 registos de cada tabela
for tabela in tabelas_silver:

    print("\n" + "=" * 100)
    print("TABELA:", tabela)
    print("=" * 100)

    df = spark.table("olist_delta." + tabela)

    print("Primeiros 5 registos:")
    display(df.limit(5))


TABELA: silver_orders
Primeiros 5 registos:


DataFrame[order_id: string, customer_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp]


TABELA: silver_order_items
Primeiros 5 registos:


DataFrame[order_id: string, order_item_id: int, product_id: string, seller_id: string, shipping_limit_date: timestamp, price: double, freight_value: double]


TABELA: silver_order_reviews
Primeiros 5 registos:


DataFrame[review_id: string, order_id: string, review_score: int, review_comment_title: string, review_comment_message: string, review_creation_date: timestamp, review_answer_timestamp: timestamp]

... (4 outputs left out)


TABELA: silver_weather_observations
Primeiros 5 registos:


DataFrame[station_id: string, state_code: string, observation_date: date, observation_time: string, precipitation: double, temperature: double, relative_humidity: double, latitude: double, longitude: double]


TABELA: silver_weather_daily_state
Primeiros 5 registos:


DataFrame[state_code: string, observation_date: date, avg_precipitation: double, avg_temperature: double, avg_relative_humidity: double, station_count: int, precipitation_coverage: double]

In [14]:
# Lista das tabelas Silver
tabelas_silver = [
    "silver_orders",
    "silver_order_items",
    "silver_order_reviews",
    "silver_sellers",
    "silver_customers",
    "silver_weather_observations",
    "silver_weather_daily_state"
]

# Mostrar os primeiros 5 registos de cada tabela
for tabela in tabelas_silver:

    print("\n" + "=" * 100)
    print("TABELA:", tabela)
    print("=" * 100)

    df = spark.table("olist_delta." + tabela)

    df.show(5, truncate=False)


TABELA: silver_orders
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|00125cb692d04887809806618a2a145f|8afb90a97ee661103014329b1bcea1a2|delivered   |2017-03-23 12:21:17     |2017-03-23 13:05:42|2017-03-27 08:58:04         |2017-04-07 15:32:47          |2017-04-20 00:00:00          |
|00130c0eee84a3d909e75bc08c5c3ca1|47d656c08a282ec5ab4c1f18303a3a30|delivered   |2018-06-11 21:17:00     |2018-06-12 0